# Structural analysis: IRFs + FEVD (Cholesky)\n\nThis notebook computes Cholesky-identified impulse responses (IRFs) and forecast error\nvariance decompositions (FEVD) from posterior draws.\n\nFor simplicity, we use a conjugate NIW BVAR on the bundled example dataset.\n

In [ ]:
import numpy as np\nimport pandas as pd\n\nfrom srvar import Dataset\nfrom srvar.api import fit\nfrom srvar.analysis.fevd import fevd_cholesky\nfrom srvar.analysis.irf import irf_cholesky\nfrom srvar.spec import ModelSpec, PriorSpec, SamplerConfig\n\ndf = pd.read_csv("../data/example.csv")\ndf["date"] = pd.to_datetime(df["date"])\n\nvalues = df[["r", "y"]].to_numpy(dtype=float)\nds = Dataset.from_arrays(values=values, variables=["r", "y"], time_index=df["date"])\n\nmodel = ModelSpec(p=2, include_intercept=True)\nprior = PriorSpec.niw_default(k=1 + ds.N * model.p, n=ds.N)\nsampler = SamplerConfig(draws=200, burn_in=0, thin=1)\n\nfit_res = fit(ds, model, prior, sampler, rng=np.random.default_rng(0))\nfit_res\n

In [ ]:
irf = irf_cholesky(fit_res, horizons=12, draws=200, rng=np.random.default_rng(1))\nfevd = fevd_cholesky(fit_res, horizons=12, draws=200, rng=np.random.default_rng(2))\n\nprint("IRF draws:", irf.draws.shape)\nprint("FEVD draws:", fevd.draws.shape)\n

## Labeled outputs (`xarray`)\n\nConvert IRF/FEVD results to labeled `xarray.Dataset` objects for safer downstream work.\n

In [ ]:
try:\n    from srvar.xarray import fevd_to_xarray, irf_to_xarray\n\n    ds_irf = irf_to_xarray(irf)\n    ds_fevd = fevd_to_xarray(fevd)\n    print(ds_irf)\n    print(ds_fevd)\nexcept ImportError as e:\n    print(e)\n